In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# 1. LAZY LOADING (Efficiency First)
# Using scan_csv allows Polars to optimize the query plan before execution
ratings_lf = pl.scan_csv("/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation/raw_data/user_ratings.csv")
games_lf = pl.scan_csv("/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation/raw_data/games.csv")

# 2. CALCULATING CORE METRICS
# We wrap these in a single collect() to minimize data passes
stats = ratings_lf.select([
    pl.col("Username").n_unique().alias("n_users"),
    pl.col("BGGId").n_unique().alias("n_items"),
    pl.len().alias("n_interactions")
]).collect()

n_users = stats["n_users"][0]
n_items = stats["n_items"][0]
n_interactions = stats["n_interactions"][0]

# 3. DENSITY & AVG INTERACTIONS
# Math: Density = Interactions / (Users * Items)
density = n_interactions / (n_users * n_items)
avg_interactions_per_user = n_interactions / n_users

print(f"--- Dataset Statistics ---")
print(f"Total Users:        {n_users:,}")
print(f"Total Items (Games): {n_items:,}")
print(f"Total Interactions:  {n_interactions:,}")
print(f"Density:             {density:.6%}")
print(f"Avg Recs per User:   {avg_interactions_per_user:.2f}")

# 4. ANALYZING COLD-START CHARACTERISTICS
# Distribution of ratings per user (to see the "Long Tail")
user_counts = (
    ratings_lf.group_by("Username")
    .agg(pl.len().alias("count"))
    .collect()
)

print(f"\n--- Cold-Start Analysis ---")
print(f"Users with only 1 rating: {user_counts.filter(pl.col('count') == 1).height:,}")
print(f"Median ratings per user:  {user_counts['count'].median()}")

# 5. METADATA INVENTORY
# Checking what features we have in the games file
metadata_cols = games_lf.columns
print(f"\n--- Available Metadata ---")
print(f"Columns: {metadata_cols}")